# Pixel Art via Redução de Cores (K-means) e Downsampling

**Disciplina:** Computação Científica e Análise de Dados

## Motivação e Modelagem

Transformar uma imagem em *pixel art* envolve resolver dois problemas de natureza distinta:

1. **Redução de resolução (downsampling):** dividir a imagem original em blocos e representar cada bloco por um único "pixel grande" (a média das cores do bloco).
2. **Redução de paleta de cores (quantização de cor):** limitar a imagem a apenas `k` cores, escolhidas de forma que representem bem todas as cores originais.

O segundo problema é exatamente um problema de **clusterização**: cada pixel é um ponto em $\mathbb{R}^3$ (canais R, G, B), e queremos agrupá-los em $k$ clusters que minimizem a soma dos quadrados das distâncias de cada ponto ao centro do seu cluster — ou seja, um problema de **Reconhecimento de Padrões** resolvido via **K-means**, conforme a Seção 2.7.6.2 do CoCADa.

Como bônus, comparamos com uma abordagem alternativa de compressão via **SVD** (Decomposição em Valores Singulares), aplicada canal a canal.

Etapas:

1. Setup e imagem de teste
2. Downsampling (blocos → pixels grandes)
3. K-means na paleta de cores
4. Escolha de k (método do cotovelo)
5. Reconstrução e comparação visual
6. (Bônus) SVD para compressão por posto baixo


## Etapa 1 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans

%matplotlib inline
np.random.seed(42)

### Imagem de teste (sintética)

Usamos primeiro uma imagem **sintética** (formas geométricas com cores conhecidas) para validar
o pipeline — como sabemos exatamente quais cores existem, conseguimos verificar se o K-means
está encontrando os clusters certos antes de aplicar numa foto real.

In [ ]:
def gerar_imagem_sintetica(tamanho=240):
    """Gera uma imagem de teste com blocos de cores conhecidas + um degradê,
    para termos controle sobre o que o K-means deveria encontrar."""
    img = np.zeros((tamanho, tamanho, 3), dtype=np.uint8)

    cores = [
        (230, 60, 60),    # vermelho
        (60, 140, 230),   # azul
        (60, 200, 100),   # verde
        (240, 200, 40),   # amarelo
    ]
    meio = tamanho // 2
    img[:meio, :meio] = cores[0]
    img[:meio, meio:] = cores[1]
    img[meio:, :meio] = cores[2]
    img[meio:, meio:] = cores[3]

    for i in range(tamanho):
        for j in range(tamanho):
            fator = (i + j) / (2 * tamanho)
            ruido = (np.random.randn(3) * 10)
            img[i, j] = np.clip(img[i, j] * (1 - 0.3 * fator) + ruido, 0, 255)

    return Image.fromarray(img)

imagem_teste = gerar_imagem_sintetica()
plt.figure(figsize=(4, 4))
plt.imshow(imagem_teste)
plt.title(f"Imagem de teste — shape {np.array(imagem_teste).shape}")
plt.axis('off')
plt.show()

### Imagem real (upload interativo)

A célula abaixo abre um seletor de arquivos direto no Colab. Toda vez que quiser testar
com uma imagem nova, basta rodar essa célula de novo e escolher o arquivo — o nome é
capturado automaticamente, sem precisar editar código.

In [ ]:
from google.colab import files

uploaded = files.upload()  # abre a caixa de seleção de arquivo
nome_arquivo = list(uploaded.keys())[0]  # pega o nome do arquivo escolhido

imagem = Image.open(nome_arquivo).convert('RGB')
imagem_array = np.array(imagem)

plt.figure(figsize=(5, 5))
plt.imshow(imagem)
plt.title(f"{nome_arquivo} — shape {imagem_array.shape}")
plt.axis('off')
plt.show()

print("Dimensões:", imagem_array.shape)

## Etapa 2 — Downsampling (redução de resolução)

In [ ]:
def downsample(img_array, tamanho_bloco):
    """Divide a imagem em blocos NxN e substitui cada bloco pela média das cores."""
    altura, largura, canais = img_array.shape

    altura_ok = (altura // tamanho_bloco) * tamanho_bloco
    largura_ok = (largura // tamanho_bloco) * tamanho_bloco
    img_cortada = img_array[:altura_ok, :largura_ok]

    blocos = img_cortada.reshape(
        altura_ok // tamanho_bloco, tamanho_bloco,
        largura_ok // tamanho_bloco, tamanho_bloco,
        canais
    )
    medias = blocos.mean(axis=(1, 3))

    resultado = np.repeat(np.repeat(medias, tamanho_bloco, axis=0), tamanho_bloco, axis=1)
    return resultado.astype(np.uint8)


TAMANHO_BLOCO = 30  # ajuste conforme o tamanho da sua imagem
img_downsampled = downsample(imagem_array, tamanho_bloco=TAMANHO_BLOCO)

fig, axs = plt.subplots(1, 2, figsize=(9, 4.5))
axs[0].imshow(imagem_array)
axs[0].set_title("Original")
axs[0].axis('off')
axs[1].imshow(img_downsampled, interpolation='nearest')
axs[1].set_title(f"Downsampled (bloco={TAMANHO_BLOCO})")
axs[1].axis('off')
plt.show()

## Etapa 3 — Redução de paleta de cores (K-means)

In [ ]:
def quantizar_cores(img_array, k, amostra=10000):
    """Reduz a imagem para k cores usando K-means. Treina numa amostra de pixels
    (mais rápido) e aplica o resultado em toda a imagem."""
    altura, largura, canais = img_array.shape
    pixels = img_array.reshape(-1, canais).astype(np.float64)

    n_pixels = pixels.shape[0]
    if n_pixels > amostra:
        idx = np.random.choice(n_pixels, amostra, replace=False)
        pixels_treino = pixels[idx]
    else:
        pixels_treino = pixels

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(pixels_treino)

    labels_completo = kmeans.predict(pixels)
    paleta = kmeans.cluster_centers_.astype(np.uint8)

    img_quantizada = paleta[labels_completo].reshape(altura, largura, canais)
    return img_quantizada, paleta


K = 6  # número de cores da pixel art — ajuste depois de ver o cotovelo (Etapa 4)
img_kmeans, paleta = quantizar_cores(img_downsampled, k=K)

fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
axs[0].imshow(imagem_array)
axs[0].set_title("Original")
axs[0].axis('off')
axs[1].imshow(img_downsampled, interpolation='nearest')
axs[1].set_title("Downsampled")
axs[1].axis('off')
axs[2].imshow(img_kmeans.astype(np.uint8), interpolation='nearest')
axs[2].set_title(f"Pixel Art (bloco={TAMANHO_BLOCO}, k={K})")
axs[2].axis('off')
plt.show()

print("Paleta de cores encontrada (RGB):")
print(paleta)

## Etapa 4 — Escolha de k (Método do Cotovelo)

In [ ]:
def calcular_inercias(img_array, k_valores, amostra=10000):
    """Calcula a inércia (soma dos quadrados das distâncias intra-cluster) para cada k."""
    pixels = img_array.reshape(-1, 3).astype(np.float64)
    n_pixels = pixels.shape[0]
    if n_pixels > amostra:
        idx = np.random.choice(n_pixels, amostra, replace=False)
        pixels_treino = pixels[idx]
    else:
        pixels_treino = pixels

    inercias = []
    for k in k_valores:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(pixels_treino)
        inercias.append(kmeans.inertia_)
    return inercias


k_valores = range(1, 13)
inercias = calcular_inercias(img_downsampled, k_valores)

plt.figure(figsize=(6, 4))
plt.plot(list(k_valores), inercias, marker='o')
plt.xlabel("k (número de cores)")
plt.ylabel("Inércia (erro quadrático total)")
plt.title("Método do Cotovelo")
plt.grid(alpha=0.3)
plt.show()

## Etapa 5 — Comparação visual entre diferentes valores de k

In [ ]:
k_testados = [2, 3, 4, 6, 8, 12]

fig, axs = plt.subplots(2, 3, figsize=(12, 9))
axs = axs.flatten()

for ax, k in zip(axs, k_testados):
    img_k, _ = quantizar_cores(img_downsampled, k=k)
    ax.imshow(img_k.astype(np.uint8), interpolation='nearest')
    ax.set_title(f"k = {k}")
    ax.axis('off')

plt.suptitle(f"Pixel Art — variando k (bloco = {TAMANHO_BLOCO})", fontsize=14)
plt.tight_layout()
plt.show()

## Etapa 6 (Bônus) — Compressão por SVD (posto baixo) vs Quantização por K-means

Enquanto o K-means reduz o número de cores mantendo a resolução espacial cheia,
o SVD reduz a complexidade estrutural da imagem, aproximando-a por uma matriz
de posto baixo — sem, necessariamente, limitar a quantidade de cores.

| | K-means | SVD |
|---|---|---|
| O que reduz | Variedade de **cor** | Variedade **espacial/estrutural** |
| Resultado visual | Blocos sólidos (posterização) | Suavização/borrão |
| Critério de escolha do parâmetro | Cotovelo da inércia | Queda dos valores singulares |


In [ ]:
def compressao_svd(img_array, posto):
    """Aplica SVD em cada canal RGB separadamente e reconstrói usando
    apenas os `posto` maiores valores singulares (aproximação de posto baixo)."""
    altura, largura, canais = img_array.shape
    img_reconstruida = np.zeros_like(img_array, dtype=np.float64)

    for c in range(canais):
        canal = img_array[:, :, c].astype(np.float64)
        U, S, Vt = np.linalg.svd(canal, full_matrices=False)

        U_r = U[:, :posto]
        S_r = S[:posto]
        Vt_r = Vt[:posto, :]

        img_reconstruida[:, :, c] = U_r @ np.diag(S_r) @ Vt_r

    return np.clip(img_reconstruida, 0, 255).astype(np.uint8)


postos_testados = [1, 3, 5, 15, 50, min(imagem_array.shape[0], imagem_array.shape[1])]

fig, axs = plt.subplots(2, 3, figsize=(12, 9))
axs = axs.flatten()

for ax, r in zip(axs, postos_testados):
    img_svd = compressao_svd(imagem_array, posto=r)
    ax.imshow(img_svd)
    ax.set_title(f"posto = {r}")
    ax.axis('off')

plt.suptitle("Compressão SVD — variando o posto (rank)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
img_kmeans_final, _ = quantizar_cores(img_downsampled, k=K)
img_svd_final = compressao_svd(imagem_array, posto=10)

fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
axs[0].imshow(imagem_array)
axs[0].set_title("Original")
axs[0].axis('off')
axs[1].imshow(img_kmeans_final.astype(np.uint8), interpolation='nearest')
axs[1].set_title(f"K-means (pixel art, k={K})")
axs[1].axis('off')
axs[2].imshow(img_svd_final)
axs[2].set_title("SVD (posto=10)")
axs[2].axis('off')
plt.savefig('comparacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
canal_vermelho = imagem_array[:, :, 0].astype(np.float64)
_, S, _ = np.linalg.svd(canal_vermelho, full_matrices=False)

plt.figure(figsize=(6, 4))
plt.plot(S[:50], marker='o', markersize=3)
plt.xlabel("Índice do valor singular")
plt.ylabel("Magnitude")
plt.title("Valores singulares do canal Vermelho (primeiros 50)")
plt.yscale('log')
plt.grid(alpha=0.3)
plt.show()

## Conclusões

- O método do cotovelo indicou `k` entre 3 e 4 como ponto de menor ganho marginal, mas
  `k=6` foi escolhido como melhor equilíbrio entre simplicidade e fidelidade visual.
- A compressão por SVD em postos baixos apresenta franjas cromáticas nas bordas, resultado
  de aplicar a decomposição independentemente em cada canal RGB.
- K-means e SVD reduzem redundâncias em "eixos" diferentes da imagem: **cor** vs.
  **estrutura espacial**.